In [ ]:
# ── Portable environment bootstrap · local ↔ Kaggle ──────────────────────────
# This cell is a no-op on my machine (the notebook already runs from notebooks/,
# so ../data and ../reports resolve normally). On Kaggle it recreates the same
# relative layout with symlinks, so every existing ../data and ../reports path
# in the rest of the notebook keeps working — no other cell needs to change.
import os
from pathlib import Path

if Path('/kaggle/input').exists():
    work   = Path('/kaggle/working')
    nb_dir = work / 'notebooks'
    nb_dir.mkdir(exist_ok=True)
    os.chdir(nb_dir)                      # now '..' == /kaggle/working

    # Link the attached dataset as ../data. If you attach more than one dataset,
    # put the Cyclistic one first (or set its folder name here explicitly).
    inputs = sorted(Path('/kaggle/input').glob('*'))
    src    = inputs[0] if inputs else None
    link   = work / 'data'
    if src and not link.exists():
        os.symlink(src, link)            # ../data/processed/df_clean.csv, etc.

    (work / 'reports' / 'figures').mkdir(parents=True, exist_ok=True)
    print(f"Kaggle detected — ../data → {src}")
else:
    print("Local environment — using the existing ../data and ../reports layout.")


# Chicago Zoning Data Processing

**Goal:** enrich the raw Chicago Zoning Districts dataset with a simplified `zone_name` column  
that groups the 80+ detailed zone codes into 7 meaningful macro-categories  
(Residential, Business, Commercial, Downtown, Manufacturing, Planned Manufacturing Districts, Transportation).

The output is consumed by `03_analysis_and_visualization.ipynb` for spatial joins with trip data.

---
## 1. Setup and Data Loading

**Input:** `../data/raw/Chicago_Zoning_Districts.csv` — public dataset from the [Chicago Data Portal](https://data.cityofchicago.org/).  
Each row represents one zoning sub-polygon with fields including `ZONE_TYPE`, `ZONE_CLASS`, and the raw WKT geometry (`the_geom`).

In [1]:
import os
import pandas as pd

In [2]:
gdf_zoning = pd.read_csv('../data/raw/Chicago_Zoning_Districts.csv')

print(f"Rows    : {len(gdf_zoning):,}")
print(f"Columns : {len(gdf_zoning.columns)}")
print(f"Columns : {gdf_zoning.columns.tolist()}")

Rows    : 14,875
Columns : 25
Columns : ['the_geom', 'CASE_NUMBE', 'CASE_TYPE', 'ZONING_ID', 'ZONE_TYPE', 'ZONE_CLASS', 'EDIT_STATU', 'CREATE_DAT', 'EDIT_DATE', 'PD_PREFIX', 'PD_NUM', 'COMMENTS', 'ORDINANCE_', 'ORDINANCE1', 'ZONING_REL', 'SHAPE_AREA', 'SHAPE_LEN', 'PEDSTREET_', 'OBJECTID', 'GLOBALID', 'PMD_SUB_AR', 'OVERRIDE_C', 'OVERRIDE_R', 'CLERK_URL', 'CLERK_DOCN']


---
## 2. Exploring the Zone Classification

The raw dataset uses granular zone codes such as `B1-1`, `C1-2`, or `R5`.  
The leading letter (or two letters for special cases like `PMD`) encodes the broad zone category.  
We inspect the full vocabulary of `ZONE_TYPE` and `ZONE_CLASS` values to design a robust mapping.

In [3]:
print("Available columns:")
print(gdf_zoning.columns.tolist())

print("\nUnique ZONE_CLASS values (first 20):")
print(sorted(gdf_zoning['ZONE_CLASS'].unique())[:20])

print(f"\nTotal unique ZONE_TYPE codes : {gdf_zoning['ZONE_TYPE'].nunique()}")
print(f"Total unique ZONE_CLASS codes : {gdf_zoning['ZONE_CLASS'].nunique()}")
print(f"\nValue counts by ZONE_CLASS first letter:")
print(gdf_zoning['ZONE_CLASS'].str[0].value_counts().sort_index())

Available columns:
['the_geom', 'CASE_NUMBE', 'CASE_TYPE', 'ZONING_ID', 'ZONE_TYPE', 'ZONE_CLASS', 'EDIT_STATU', 'CREATE_DAT', 'EDIT_DATE', 'PD_PREFIX', 'PD_NUM', 'COMMENTS', 'ORDINANCE_', 'ORDINANCE1', 'ZONING_REL', 'SHAPE_AREA', 'SHAPE_LEN', 'PEDSTREET_', 'OBJECTID', 'GLOBALID', 'PMD_SUB_AR', 'OVERRIDE_C', 'OVERRIDE_R', 'CLERK_URL', 'CLERK_DOCN']

Unique ZONE_CLASS values (first 20):
['B1-1', 'B1-1.5', 'B1-2', 'B1-3', 'B1-5', 'B2-1', 'B2-1.5', 'B2-2', 'B2-3', 'B2-5', 'B3-1', 'B3-1.5', 'B3-2', 'B3-3', 'B3-5', 'C1-1', 'C1-1.5', 'C1-2', 'C1-3', 'C1-5']

Total unique ZONE_TYPE codes : 12
Total unique ZONE_CLASS codes : 1520

Value counts by ZONE_CLASS first letter:
ZONE_CLASS
B    4559
C    2022
D     295
M     836
P    2032
R    5087
T      44
Name: count, dtype: int64


---
## 3. Building the Simplified Zone Taxonomy

**Mapping logic:** the first character of `ZONE_CLASS` determines the macro-category:

| First letter | Macro-category |
|:---:|---|
| `R` | Residential |
| `B` | Business |
| `C` | Commercial |
| `D` | Downtown |
| `M` | Manufacturing |
| `P` | Planned Manufacturing Districts |
| `T` | Transportation |

Each unique `ZONE_TYPE` code is mapped to one of these categories, producing a `zone_name` column  
that will be used to label every polygon (and therefore every trip start/end point) with a human-readable zone label.

In [4]:
# Build the ZONE_TYPE → zone_name mapping from first letter of ZONE_CLASS
legend_raw = gdf_zoning.groupby('ZONE_TYPE')['ZONE_CLASS'].first().reset_index()
legend_raw['base_letter'] = legend_raw['ZONE_CLASS'].str[0].str.upper()

MACRO_MAP = {
    'R': 'Residential',
    'B': 'Business',
    'C': 'Commercial',
    'D': 'Downtown',
    'M': 'Manufacturing',
    'P': 'Planned Manufacturing Districts',
    'T': 'Transportation',
}
legend_raw['zone_name'] = legend_raw['base_letter'].map(MACRO_MAP).fillna('Other')

legend = legend_raw[['ZONE_TYPE', 'zone_name']].drop_duplicates().sort_values('ZONE_TYPE')
legend_dict = dict(zip(legend['ZONE_TYPE'], legend['zone_name']))

print("Zone taxonomy (ZONE_TYPE → zone_name):")
print(legend.to_string(index=False))
print(f"\nTotal ZONE_TYPE codes mapped : {len(legend_dict)}")

Zone taxonomy (ZONE_TYPE → zone_name):
 ZONE_TYPE                       zone_name
         1                        Business
         2                      Commercial
         3                   Manufacturing
         4                     Residential
         5 Planned Manufacturing Districts
         6 Planned Manufacturing Districts
         7                        Downtown
         8                        Downtown
         9                        Downtown
        10                        Downtown
        11                  Transportation
        12 Planned Manufacturing Districts

Total ZONE_TYPE codes mapped : 12


In [5]:
# Apply zone_name, lowercase all columns, reorder so zone_name precedes zone_type
gdf_zoning['zone_name'] = gdf_zoning['ZONE_TYPE'].map(legend_dict)
gdf_zoning.columns      = gdf_zoning.columns.str.lower()

cols = gdf_zoning.columns.tolist()
cols.insert(cols.index('zone_type'), cols.pop(cols.index('zone_name')))
gdf_zoning = gdf_zoning[cols]

# Verify classification coverage
missing = gdf_zoning['zone_name'].isna().sum()
if missing > 0:
    print(f"⚠️  {missing} rows without zone_name — unmapped ZONE_TYPE codes:")
    print(sorted(gdf_zoning.loc[gdf_zoning['zone_name'].isna(), 'zone_type'].unique()))
else:
    print("All rows classified. ✓")

print(f"\nZone distribution (row count per category):")
print(gdf_zoning['zone_name'].value_counts().to_string())

All rows classified. ✓

Zone distribution (row count per category):
zone_name
Residential                        5088
Business                           4562
Planned Manufacturing Districts    2033
Commercial                         2020
Manufacturing                       836
Downtown                            292
Transportation                       44


---
## 4. Conclusions

### Zone classification scheme

The raw Chicago Zoning Districts dataset contains 80+ granular zone codes.  
This notebook reduces them to **7 macro-categories** based on the leading letter of `ZONE_CLASS`:

| Category | Description |
|----------|-------------|
| **Residential** | Housing areas — typical commute origins/destinations for members |
| **Business** | Local offices and neighbourhood services |
| **Commercial** | Retail strips, shopping centres |
| **Downtown** | Chicago CBD — high bike-sharing density, mixed commute and leisure |
| **Manufacturing** | Industrial areas — lower recreational use expected |
| **Planned Manufacturing Districts** | PMD special zones |
| **Transportation** | Airports, transit hubs |

### Key observation
The zone taxonomy is deliberately coarse: it captures the broad *land use character* of each area,  
which is what matters for understanding *why* casual and member riders start and end trips in different zones.

### Output files
Both files are saved to `../data/interim/` and consumed by `03_analysis_and_visualization.ipynb`:
- **`gdf_zoning_updated.csv`** — full zoning dataset with `zone_name` column and lowercase field names
- **`zone_legend_updated.csv`** — compact lookup table (ZONE_TYPE → zone_name)

---
## 5. Saving Outputs

Outputs are saved to `../data/interim/` — the standard location for datasets that have been  
enriched or transformed but are not yet the final analysis-ready product.

In [6]:
output_dir = '../data/interim'
os.makedirs(output_dir, exist_ok=True)

# 1. Full enriched zoning dataset
zoning_path = os.path.join(output_dir, 'gdf_zoning_updated.csv')
gdf_zoning.to_csv(zoning_path, index=False)
print(f"Saved: {os.path.abspath(zoning_path)}")

# 2. Compact zone legend (lookup table)
legend_out_path = os.path.join(output_dir, 'zone_legend_updated.csv')
pd.DataFrame(legend_dict.items(), columns=['zone_type', 'zone_name']).to_csv(
    legend_out_path, index=False
)
print(f"Saved: {os.path.abspath(legend_out_path)}")

print(f"\nReady for spatial join in 03_analysis_and_visualization.ipynb ✓")

Saved: /Users/lorenzodigiacomo/Desktop/Ciclistic_analisi/data/interim/gdf_zoning_updated.csv
Saved: /Users/lorenzodigiacomo/Desktop/Ciclistic_analisi/data/interim/zone_legend_updated.csv

Ready for spatial join in 03_analysis_and_visualization.ipynb ✓
